In [1]:
import polars as pl
import polars.selectors as cs
import duckdb
import orbital
import sqlglot
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from clean_sql import clean_sql

In [2]:
# hide ibis FutureWarning triggered by orbital
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# get data
with duckdb.connect('../dev.duckdb') as con:
    df_feat = con.table("raw_feat").pl()
    df_targ = con.table("raw_targ").pl()

# split train + test sets 
## assign these in database so they persist; could be useful for later analysis
df_feat_train = df_feat.filter( pl.col('cat_train_test') == pl.lit('Train') ).drop('cat_train_test')
df_feat_test  = df_feat.filter( pl.col('cat_train_test') == pl.lit('Test') ).drop('cat_train_test')

# prep for modeling (order, split)
df = df_targ.join(df_feat_train, how = "inner", on = "customer_id")
X = df.drop("customer_id", "churn")
y = df.select("churn").to_numpy().ravel()

In [4]:
# build pipeline(s)

## feature pipeline does OneHotEncoding on all string columns (all are low/known cardinality)
## orbital can create some very verbose variable names (for uniqueness) so we clean those up some
cols_str = X.select( cs.string() ).columns
onho_enc = ('oh', OneHotEncoder(sparse_output = False), cols_str)
ppl_feat = Pipeline([
  ("encoder", ColumnTransformer([onho_enc], remainder='passthrough'))
]).set_output(transform="polars")
X_tran = ppl_feat.fit_transform(X, y)
X_tran.columns = [c.replace(' ','_').replace('-','_').replace('(','').replace(')','') for c in X_tran.columns]

## training pipeline fits actual random forest model
ppl_pred = Pipeline([
  ("prep", ColumnTransformer([], remainder='passthrough')),
  ("pred", RandomForestClassifier(max_depth=3, n_estimators=100, random_state=123))
])
ppl_pred.fit(X_tran, y)

## save out predictions for comparison
df_preds_py = pl.DataFrame({
  'customer_id': df.select('customer_id').to_numpy().ravel(),
  'pred': ppl_pred.predict_proba(X_tran)[:,1]
})
df_preds_py.write_csv('preds_py.csv')

This is the part where you actually check if your model is any good. That is not my problem at the moment. =)

In [5]:
# convert to orbital

tbl = "TBL_REF" # placeholder replaced in cleaning

## creating mapping of source data types to orbital types 
type_map = {
    pl.String:orbital.types.StringColumnType(),
    pl.Int32:orbital.types.Int32ColumnType(),
    pl.Float64:orbital.types.DoubleColumnType()
}
dict_feat = {e: type_map.get(t) for e, t in zip(X.columns, X.dtypes)}
dict_pred = {e: type_map.get(t) for e, t in zip(X_tran.columns, X_tran.dtypes)}

## features
orb_ppl_feat = orbital.parse_pipeline(ppl_feat, features=dict_feat)
sql_raw_feat = orbital.export_sql(tbl, orb_ppl_feat, dialect="duckdb")

## scoring
orb_ppl_pred = orbital.parse_pipeline(ppl_pred, features=dict_pred)
sql_raw_pred = orbital.export_sql(tbl, orb_ppl_pred, dialect="duckdb")

In [6]:
# clean up resulting SQL

## map long orbital names into simpler names sklearn produces
dict_renm = dict(zip(
    [e.alias for e in sqlglot.parse_one(sql_raw_feat)],
    X_tran.columns
))

## use custom converters
sql_fmt_feat = clean_sql(sql_raw_feat, 'raw_feat', model_version = "1.0", col_id = 'customer_id', cols_renm = dict_renm)
sql_fmt_pred = clean_sql(sql_raw_pred, 'prep_feat', model_version = "1.0", col_id = 'customer_id')

In [7]:
# write to dbt model

with open("../models/churn_model/prep_feat.sql", "w") as file:

    config = '{{ config( materialized="view") }}'
    file.writelines([config, '\n\n', sql_fmt_feat])

with open("../models/churn_model/pred_churn.sql", "w") as file:
    
    config = '{{ config( materialized="view") }}'
    file.writelines([config, '\n\n', sql_fmt_pred])